In [2]:
import torch
import torch.nn as nn

#模拟一张1*28*28的单通道图像
x = torch.randn(1,1,28,28)
print("0.原始输入形状：",x.shape)

conv1 = nn.Conv2d(in_channels=1,out_channels=6,kernel_size=5,stride=1,padding=0)
pool = nn.MaxPool2d(kernel_size=2,stride =2)

out1 = conv1(x)
print(f"out_1形状：{out1.shape}")
out1_pool = pool(out1)
print(f"out_1pool形状：{out1_pool.shape}")

conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1, padding=0)
out2 = conv2(out1_pool)
print("3. Conv2 输出 (预期 [1, 16, 8, 8]):", out2.shape)
out2_pool = pool(out2)
print("4. Pool2 输出 (预期 [1, 16, 4, 4]):", out2_pool.shape)

flat = out2_pool.flatten(start_dim=1)
print(f"flatten输出：{flat.shape}")
fc = nn.Linear(256,10)
logits = fc(flat)
print(f"logits形状：{logits.shape}")

0.原始输入形状： torch.Size([1, 1, 28, 28])
out_1形状：torch.Size([1, 6, 24, 24])
out_1pool形状：torch.Size([1, 6, 12, 12])
3. Conv2 输出 (预期 [1, 16, 8, 8]): torch.Size([1, 16, 8, 8])
4. Pool2 输出 (预期 [1, 16, 4, 4]): torch.Size([1, 16, 4, 4])
flatten输出：torch.Size([1, 256])
logits形状：torch.Size([1, 10])


In [3]:
# 模拟：Batch=2, 源句子长度=7, 词向量维度=16, 隐层维度=32
src_seq = torch.randn(7, 2, 16)   # [Seq_len, Batch, Embed_dim]

# 1. 编码器 (Encoder)
encoder_gru = nn.GRU(input_size=16, hidden_size=32, num_layers=1)
enc_outputs, enc_hidden = encoder_gru(src_seq)
print("Encoder 全部步输出 (包含每个时序的语义):", enc_outputs.shape)  # [7, 2, 32]
print("Encoder 最终隐状态 (交接给解码器的上下文):", enc_hidden.shape)   # [1, 2, 32]

# 2. 解码器 (Decoder 单步解码)
# 解码器第一步的输入是起始符 <BOS>，状态直接接收 enc_hidden
dec_gru = nn.GRU(input_size=16, hidden_size=32, num_layers=1)
dec_input = torch.randn(1, 2, 16) # 仅输入 1 个词 [1, Batch, Embed_dim]

dec_output, dec_hidden = dec_gru(dec_input, enc_hidden)
print("Decoder 第 1 步输出:", dec_output.shape) # [1, 2, 32]
print("Decoder 更新后的隐状态:", dec_hidden.shape) # [1, 2, 32]

Encoder 全部步输出 (包含每个时序的语义): torch.Size([7, 2, 32])
Encoder 最终隐状态 (交接给解码器的上下文): torch.Size([1, 2, 32])
Decoder 第 1 步输出: torch.Size([1, 2, 32])
Decoder 更新后的隐状态: torch.Size([1, 2, 32])
